## Demand Forecasting — Moving Average
7-day and 30-day moving average demand signals. No extra installs required.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Read Daily Demand Gold Table

In [0]:
df = spark.read.table("databricks_cat.gold.DailyDemand")
df.printSchema()

### 7-Day and 30-Day Moving Average per Product

In [0]:
window_7d  = Window.partitionBy("product_id").orderBy("order_date").rowsBetween(-6, 0)
window_30d = Window.partitionBy("product_id").orderBy("order_date").rowsBetween(-29, 0)

df_ma = (
    df
    .withColumn("ma_7day_units",    avg("units_sold").over(window_7d))
    .withColumn("ma_30day_units",   avg("units_sold").over(window_30d))
    .withColumn("ma_7day_revenue",  avg("daily_revenue").over(window_7d))
    .withColumn("ma_30day_revenue", avg("daily_revenue").over(window_30d))
    # Trend: 7-day MA vs 30-day MA with 5% threshold
    .withColumn("demand_trend",
        when(col("ma_7day_units") > col("ma_30day_units") * 1.05, lit("RISING"))
        .when(col("ma_7day_units") < col("ma_30day_units") * 0.95, lit("FALLING"))
        .otherwise(lit("STABLE"))
    )
)
df_ma.display()

### 30-Day Simple Forecast per Product
Project next 30 days based on the trailing 30-day moving average.

In [0]:
# Get the latest MA values per product (most recent date)
window_latest = Window.partitionBy("product_id").orderBy(desc("order_date"))

df_latest_ma = (
    df_ma
    .withColumn("rn", row_number().over(window_latest))
    .filter(col("rn") == 1)
    .select("product_id", "product_name", "category", "brand",
            "order_date", "ma_30day_units", "ma_30day_revenue")
)

# Generate 30 future dates per product using cross join with a date range
future_days = spark.range(1, 31).withColumn("day_offset", col("id").cast("int")).drop("id")

df_forecast = (
    df_latest_ma
    .crossJoin(future_days)
    .withColumn("forecast_date",        date_add(col("order_date"), col("day_offset")))
    .withColumn("forecasted_units",     round(col("ma_30day_units"), 0))
    .withColumn("forecasted_revenue",   round(col("ma_30day_revenue"), 2))
    .withColumn("forecast_lower_bound", round(col("ma_30day_units") * 0.85, 0))
    .withColumn("forecast_upper_bound", round(col("ma_30day_units") * 1.15, 0))
    .select("product_id", "product_name", "category", "brand",
            "forecast_date", "forecasted_units", "forecasted_revenue",
            "forecast_lower_bound", "forecast_upper_bound")
)
df_forecast.display()

### Save Moving Average Signals → Gold

In [0]:
(
    df_ma.write.format("delta")
    .mode("overwrite")
    .option("path", "abfss://gold@databricksetestorage.dfs.core.windows.net/DemandForecast_MA")
    .saveAsTable("databricks_cat.gold.DemandForecast_MA")
)
print("✅ DemandForecast_MA written.")

### Save 30-Day Forecast → Gold

In [0]:
(
    df_forecast.write.format("delta")
    .mode("overwrite")
    .option("path", "abfss://gold@databricksetestorage.dfs.core.windows.net/DemandForecast_30D")
    .saveAsTable("databricks_cat.gold.DemandForecast_30D")
)
print("✅ DemandForecast_30D written.")

In [0]:
%sql
SELECT product_name, category, forecast_date,
       forecasted_units, forecast_lower_bound, forecast_upper_bound
FROM databricks_cat.gold.DemandForecast_30D
ORDER BY forecast_date ASC
LIMIT 30